# HW6: Interpretando los coeficientes de cuatro modelos con logaritmo

**Análisis de Regresión · Ingeniería en Ciencia de Datos · ITESO**

---

**Autoría:** Lambda Remiel Heredia Pérez

**Entrada:** `HW6_Log_Transformations_Interpretation.xlsx`

**Entregable:** `HW6_Log_Transformations_Interpretation_Solved.xlsx`

### De qué va esto

Un marketplace de carros usados quiere saber qué mueve el precio que la gente pide por su carro, así que alguien le ajustó cuatro regresiones a los mismos 400 carros, con el logaritmo puesto en un lugar distinto cada vez. Aquí NO tenemos los datos crudos, solo las tablas de coeficientes, entonces el rollo es leerlas bien, es decir, decidir qué variables son significativas al 95%, calcular el efecto cuando la escala lo pide y decirlo en términos de negocio (con todo y unidades, que es donde es más fácil tropezarse).

Todo lo que sale aquí termina en las celdas amarillas del Excel, con fórmulas vivas que apuntan a la celda del coeficiente (no números pegados a mano). Las funciones están en `hw6_lectura_log.py` para que esto se pueda leer de corrido.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from hw6_lectura_log import (
    N_OBS, comparar_con_excel, escribir_respuestas, exacto_vs_aprox, leer_modelos,
    mostrar, recalcular_con_excel, resolver_modelo, resumen_modelos, revisar_inferencia,
    tabla_lecturas, tabla_significancia,
)

PLANTILLA = "HW6_Log_Transformations_Interpretation.xlsx"
SALIDA = "HW6_Log_Transformations_Interpretation_Solved.xlsx"

pd.set_option("display.max_colwidth", None)

## $\color{#dda}{\text{1. Dónde trae el log cada modelo}}$

In [2]:
modelos = leer_modelos(PLANTILLA)
resumen_modelos(modelos)

modelo,y en log,x en log
Model 1 LIN-LIN,No,ninguna
Model 2 LIN-LOG,No,np.log(km_mil)
Model 3 LOG-LIN,Sí,ninguna
Model 4 LOG-LOG,Sí,"np.log(km_mil), np.log(potencia_hp)"


**Interpretación.** Yo creo que aquí está toda la trampa de la tarea, porque la misma variable cambia de papel de una hoja a otra. `km_mil` va en log en los Modelos 2 y 4 pero en niveles en el 1 y el 3, y `potencia_hp` solo va en log en el Modelo 4 (en el 2 se sigue leyendo por hp, aunque esa hoja ya traiga log en los km). `antiguedad_anios`, `num_duenos` y las dos dummies nunca llevan log. Por eso la regla se escoge coeficiente por coeficiente y NO hoja por hoja.

## $\color{#dda}{\text{2. Las tablas cuadran?}}$

Antes de creerle a las tablas recalculé cada p-value a partir de `coef` y `std err`, con una t de 393 grados de libertad (400 carros menos 7 coeficientes), y revisé que el p-value y el intervalo de 95% digan lo mismo.

In [3]:
revision = pd.concat({nombre: revisar_inferencia(m["coefs"]) for nombre, m in modelos.items()})
coinciden = (revision["sig_por_p"] == revision["sig_por_ic"]).all()

print(f"Grados de libertad: {N_OBS} - 7 = {N_OBS - 7}")
print(f"Filas revisadas: {len(revision)}")
print(f"Mayor diferencia en p-value: {(revision['p_reportado'] - revision['p_recalculado']).abs().max():.4f}")
print(f"p-value e intervalo coinciden en todas las filas: {'Sí' if coinciden else 'No'}")

Grados de libertad: 400 - 7 = 393
Filas revisadas: 28
Mayor diferencia en p-value: 0.0005
p-value e intervalo coinciden en todas las filas: Sí


**Interpretación.** Los p-values vienen con tres decimales y lo más que se alejan de los recalculados es 0.0005, o sea puro redondeo. Además el p-value y el intervalo coinciden en las 28 filas, así que podemos usar la columna `P>|t|` sin miedo. Un 0.000 en la tabla significa p < 0.0005, no que el p-value sea cero exacto.

## $\color{#dda}{\text{3. Cuáles variables son significativas}}$

Para cada coeficiente la prueba es $H_0: \beta = 0$ contra $H_1: \beta \neq 0$, y la variable es significativa si $p < 0.05$. Cada celda trae el p-value y la decisión.

In [4]:
tabla_significancia(modelos)

,Model 1 LIN-LIN,Model 2 LIN-LOG,Model 3 LOG-LIN,Model 4 LOG-LOG
variable,,,,
km_mil,0.001 (Sí),0.000 (Sí),0.000 (Sí),0.000 (Sí)
antiguedad_anios,0.000 (Sí),0.000 (Sí),0.000 (Sí),0.000 (Sí)
potencia_hp,0.000 (Sí),0.000 (Sí),0.000 (Sí),0.000 (Sí)
num_duenos,0.872 (No),0.552 (No),0.534 (No),0.313 (No)
importado,0.000 (Sí),0.000 (Sí),0.000 (Sí),0.000 (Sí)
automatica,0.047 (Sí),0.013 (Sí),0.001 (Sí),0.000 (Sí)


**Interpretación.** Cinco de las seis variables salen significativas en los cuatro modelos. La excepción es `num_duenos`, que no pasa en NINGUNO (p entre 0.313 y 0.872), entonces se escribe *Not significant* y ya, nada de leer su coeficiente como efecto. Eso tampoco prueba que el número de dueños no importe, simplemente estos modelos no encuentran evidencia una vez que lo demás se mantiene fijo.

La que se salva de panzazo es `automatica` en el Modelo 1 (p = 0.047), sí es significativa pero su intervalo [0.08, 12.72] casi toca el cero.

## $\color{#dda}{\text{4. Qué regla le toca a cada coeficiente}}$

Aquí importan dos cosas nada más, la escala de `y` y la escala de ESA variable.

| y | x | Cambio que leo | Efecto en y | Fórmula en Excel | Lleva cálculo? |
|---|---|---|---|---|---|
| Niveles | Niveles o dummy | +1 unidad | $\beta$ unidades de y | ninguna | No |
| Niveles | Log | +10% | $\beta \cdot \ln(1.1)$ unidades de y | `=B7*LN(1.1)` | Sí |
| Log | Niveles o dummy | +k unidades | $e^{k\beta} - 1$ | `=EXP(B8)-1`, `=EXP(B7*10)-1` | Sí |
| Log | Log | +10% | $1.1^{\beta} - 1$ | `=1.1^B7-1` | Sí |

Las fórmulas no salen de la nada, todas vienen de comparar el modelo antes y después del cambio. Con `y` en niveles y `x` en log, $\Delta y = \beta \, [\ln(1.1x) - \ln(x)] = \beta \ln(1.1)$. Con `y` en log y `x` en niveles, $\ln(y_1) - \ln(y_0) = k\beta$, o sea $y_1 / y_0 = e^{k\beta}$. Y con log de los dos lados, $\ln(y_1) - \ln(y_0) = \beta \ln(1.1)$, o sea $y_1 / y_0 = 1.1^{\beta}$.

La tarea deja escoger el cambio en `x`, y esto fue lo que escogí. Para las variables en log usé +10%, que es lo mismo que trae el ejemplo del Excel y además es una diferencia realista entre dos carros. En el Modelo 3 leí +10,000 km y +10 hp porque una sola unidad mueve el precio menos de 0.4%, y a quien vende un carro eso no le dice nada (ojo, el 10 va DENTRO del exponente, $e^{10\beta} - 1$, y no es lo mismo que $10 \cdot (e^{\beta} - 1)$, porque los porcentajes se componen). Para la edad me quedé con +1 año, que ya significa algo por sí solo, y en las dummies no hay nada que escoger porque el cambio siempre es pasar de la categoría de referencia a la otra.

Con las unidades hay que traer cuidado, `precio_miles` está en miles de pesos y `km_mil` en miles de km. En los Modelos 3 y 4 las fórmulas regresan una fracción porque la plantilla ya trae esas celdas con formato de porcentaje, por eso -0.0806 se ve como -8.056%.

## $\color{#dda}{\text{5. Modelo 1, LIN-LIN}}$

In [5]:
respuestas = {nombre: resolver_modelo(modelo) for nombre, modelo in modelos.items()}
mostrar(respuestas["Model 1 LIN-LIN"])

variable,p,significativa,fórmula,efecto,interpretación
km_mil,0.001,Yes,,-0.183 miles de pesos,"Si el carro trae 1,000 km más, el precio que piden baja como 183 pesos (0.183 miles de pesos), con todo lo demás igual."
antiguedad_anios,0.000,Yes,,-16.979 miles de pesos,"Si el carro tiene 1 año más, el precio que piden baja como 16,979 pesos (16.979 miles de pesos), con todo lo demás igual."
potencia_hp,0.000,Yes,,+0.765 miles de pesos,"Si el carro trae 1 hp más, el precio que piden sube como 765 pesos (0.765 miles de pesos), con todo lo demás igual."
num_duenos,0.872,No,,,Not significant
importado,0.000,Yes,,+42.602 miles de pesos,"Por un carro importado piden como 42,602 pesos (42.602 miles de pesos) más que por uno nacional, con todo lo demás igual."
automatica,0.047,Yes,,+6.401 miles de pesos,"Por un carro automático piden como 6,401 pesos (6.401 miles de pesos) más que por uno manual, con todo lo demás igual."


**Interpretación.** Aquí no hay nada que calcular y las celdas de cálculo se quedan vacías (así lo pide la tarea), porque `y` y todas las `x` están en sus unidades originales y el coeficiente ya ES el efecto. Lo único es cuidar las unidades, $\beta = -0.183$ son miles de pesos por cada mil km, es decir 183 pesos por cada 1,000 km. Un dato que me gustó, en este modelo un año de antigüedad pesa lo mismo que unos 93,000 km en el odómetro (16.979 / 0.183), o sea que dejar el carro parado tampoco te salva JAKSJASJAKSJAKSJK.

## $\color{#dda}{\text{6. Modelo 2, LIN-LOG}}$

In [6]:
mostrar(respuestas["Model 2 LIN-LOG"])

variable,p,significativa,fórmula,efecto,interpretación
np.log(km_mil),0.000,Yes,=B7*LN(1.1),-2.989 miles de pesos,"Si el kilometraje sube 10%, el precio que piden baja como 2,989 pesos (2.989 miles de pesos), con todo lo demás igual."
antiguedad_anios,0.000,Yes,,-12.940 miles de pesos,"Si el carro tiene 1 año más, el precio que piden baja como 12,940 pesos (12.940 miles de pesos), con todo lo demás igual."
potencia_hp,0.000,Yes,,+0.742 miles de pesos,"Si el carro trae 1 hp más, el precio que piden sube como 742 pesos (0.742 miles de pesos), con todo lo demás igual."
num_duenos,0.552,No,,,Not significant
importado,0.000,Yes,,+43.955 miles de pesos,"Por un carro importado piden como 43,955 pesos (43.955 miles de pesos) más que por uno nacional, con todo lo demás igual."
automatica,0.013,Yes,,+7.401 miles de pesos,"Por un carro automático piden como 7,401 pesos (7.401 miles de pesos) más que por uno manual, con todo lo demás igual."


**Interpretación.** Solo `np.log(km_mil)` lleva cálculo. Dividir $\beta$ entre 100 daría la lectura de un 1%, y para el 10% usé $\beta \ln(1.1)$. Lo chistoso es que esos 2,989 pesos son los mismos para cualquier carro, da igual si va de 20,000 a 22,000 km o de 100,000 a 110,000 km, así que cada km extra cuesta menos conforme el carro acumula km (rendimientos decrecientes, pues). `potencia_hp` sigue en niveles en esta hoja, entonces se lee por hp y en pesos, sin cálculo.

## $\color{#dda}{\text{7. Modelo 3, LOG-LIN}}$

In [7]:
mostrar(respuestas["Model 3 LOG-LIN"])

variable,p,significativa,fórmula,efecto,interpretación
km_mil,0.000,Yes,=EXP(B7*10)-1,-1.09%,"Si el carro trae 10,000 km más, el precio que piden baja como 1.09%, con todo lo demás igual."
antiguedad_anios,0.000,Yes,=EXP(B8)-1,-8.06%,"Si el carro tiene 1 año más, el precio que piden baja como 8.06%, con todo lo demás igual."
potencia_hp,0.000,Yes,=EXP(B9*10)-1,+3.97%,"Si el carro trae 10 hp más, el precio que piden sube como 3.97%, con todo lo demás igual."
num_duenos,0.534,No,,,Not significant
importado,0.000,Yes,=EXP(B11)-1,+23.84%,"Por un carro importado piden como 23.84% más que por uno nacional, con todo lo demás igual."
automatica,0.001,Yes,=EXP(B12)-1,+4.40%,"Por un carro automático piden como 4.40% más que por uno manual, con todo lo demás igual."


**Interpretación.** Con `y` en log TODOS los coeficientes significativos llevan cálculo y la respuesta sale en porcentaje, no en pesos (las dummies también, que se olvidan fácil). Donde de verdad importa es en `importado`, $\beta = 0.214$ ya se pasa del $|\beta| < 0.1$ que vimos en clase, así que leerlo como 21.4% se queda corto y lo exacto es +23.84%.

## $\color{#dda}{\text{8. Modelo 4, LOG-LOG}}$

In [8]:
mostrar(respuestas["Model 4 LOG-LOG"])

variable,p,significativa,fórmula,efecto,interpretación
np.log(km_mil),0.000,Yes,=1.1^B7-1,-0.89%,"Si el kilometraje sube 10%, el precio que piden baja como 0.89%, con todo lo demás igual."
antiguedad_anios,0.000,Yes,=EXP(B8)-1,-7.53%,"Si el carro tiene 1 año más, el precio que piden baja como 7.53%, con todo lo demás igual."
np.log(potencia_hp),0.000,Yes,=1.1^B9-1,+5.35%,"Si la potencia sube 10%, el precio que piden sube como 5.35%, con todo lo demás igual."
num_duenos,0.313,No,,,Not significant
importado,0.000,Yes,=EXP(B11)-1,+24.21%,"Por un carro importado piden como 24.21% más que por uno nacional, con todo lo demás igual."
automatica,0.000,Yes,=EXP(B12)-1,+4.53%,"Por un carro automático piden como 4.53% más que por uno manual, con todo lo demás igual."


**Interpretación.** Esta hoja mezcla dos reglas y la vdd creo que es donde es más fácil equivocarse. `np.log(km_mil)` y `np.log(potencia_hp)` son elasticidades y usan $1.1^{\beta} - 1$, mientras que `antiguedad_anios` y las dummies siguen en niveles y usan $e^{\beta} - 1$, igual que en el Modelo 3. La elasticidad de la potencia es 0.55, menor a 1, así que el precio crece menos que proporcional a la potencia (10% más de hp va con 5.35% más de precio, no con 10%).

## $\color{#dda}{\text{9. La misma variable, cuatro lecturas}}$

In [9]:
tabla_lecturas(respuestas)

,Model 1 LIN-LIN,Model 2 LIN-LOG,Model 3 LOG-LIN,Model 4 LOG-LOG
variable,,,,
km_mil,"+1,000 km: -0.183 miles de pesos",+10%: -2.989 miles de pesos,"+10,000 km: -1.09%",+10%: -0.89%
antiguedad_anios,+1 año: -16.979 miles de pesos,+1 año: -12.940 miles de pesos,+1 año: -8.06%,+1 año: -7.53%
potencia_hp,+1 hp: +0.765 miles de pesos,+1 hp: +0.742 miles de pesos,+10 hp: +3.97%,+10%: +5.35%
num_duenos,Not significant,Not significant,Not significant,Not significant
importado,importado vs nacional: +42.602 miles de pesos,importado vs nacional: +43.955 miles de pesos,importado vs nacional: +23.84%,importado vs nacional: +24.21%
automatica,automático vs manual: +6.401 miles de pesos,automático vs manual: +7.401 miles de pesos,automático vs manual: +4.40%,automático vs manual: +4.53%


**Interpretación.** Todos los efectos significativos conservan su signo en los cuatro modelos, la edad y los km bajan el precio, y la potencia, ser importado y ser automático lo suben. Los tamaños NO se pueden comparar de frente entre columnas porque cambian las unidades y también el cambio en `x` (miles de pesos contra porcentajes, ya se ve el problema). Lo que sí se ve es que entre los dos modelos con `y` en log las lecturas con el mismo cambio quedan cerca, importado +23.84% y +24.21%, automático +4.40% y +4.53%, edad -8.06% y -7.53%.

Algo que me llamó la atención es que el coeficiente de la edad baja como una cuarta parte (de -16.98 a -12.94) cuando los km entran en log, lo cual me hace pensar que edad y km comparten parte de la misma señal de desgaste, pero sin los datos crudos no lo puedo confirmar.

Y cuál modelo es el mejor? Con estas tablas no se puede saber. Haría falta ver residuales, la prueba RESET y los errores en pesos, porque la $R^2$ ni siquiera es comparable entre escalas de `y`.

## $\color{#dda}{\text{10. Por qué la fórmula exacta y no el atajo}}$

In [10]:
pd.concat(
    {nombre: exacto_vs_aprox(respuestas[nombre]) for nombre in ("Model 3 LOG-LIN", "Model 4 LOG-LOG")}
)

cambio      coef  aprox_%  \
                variable                                                        
Model 3 LOG-LIN km_mil                          +10,000 km -0.001095   -1.095   
                antiguedad_anios                    +1 año -0.083993   -8.399   
                potencia_hp                         +10 hp  0.003894    3.894   
                importado            importado vs nacional  0.213856   21.386   
                automatica            automático vs manual  0.043104    4.310   
Model 4 LOG-LOG np.log(km_mil)                        +10% -0.093329   -0.933   
                antiguedad_anios                    +1 año -0.078314   -7.831   
                np.log(potencia_hp)                   +10%  0.546554    5.466   
                importado            importado vs nacional  0.216820   21.682   
                automatica            automático vs manual  0.044299    4.430   

                                     exacto_%  brecha_pts  
                variable                                   
Model 3 LOG-LIN km_mil                 -1.089       0.006  
                antiguedad_anios       -8.056       0.343  
                potencia_hp             3.971       0.077  
                importado              23.844       2.459  
                automatica              4.405       0.094  
Model 4 LOG-LOG np.log(km_mil)         -0.886       0.048  
                antiguedad_anios       -7.533       0.299  
                np.log(potencia_hp)     5.347      -0.118  
                importado              24.212       2.530  
                automatica              4.529       0.100

**Interpretación.** El atajo aguanta bien con coeficientes chicos, la brecha se queda abajo de 0.35 puntos porcentuales, pero en `importado` se va a unos 2.5 puntos, que es justo lo que decía la regla de clase (si $|\beta| < 0.1$ se puede leer $\beta$ como porcentaje, si no hay que reportar $e^{\beta} - 1$). En el Excel usé la fórmula exacta en todos lados para no andar decidiendo cuál sí y cuál no.

## $\color{#dda}{\text{11. Llenar el Excel}}$

Las respuestas van a una copia para no tocar la plantilla, y solo se escriben las celdas amarillas. Después Excel recalcula el archivo para que las fórmulas guarden su resultado (si no, algunas vistas previas las muestran vacías) y comparo cada resultado contra el de aquí.

In [11]:
escribir_respuestas(PLANTILLA, SALIDA, respuestas)
recalcular_con_excel(SALIDA)

comparacion = comparar_con_excel(SALIDA, respuestas)
print(f"Mayor brecha entre Excel y Python: {comparacion['brecha'].max():.2e}")
comparacion

Mayor brecha entre Excel y Python: 0.00e+00


,hoja,celda,fórmula,python,excel,brecha
0,Model 2 LIN-LOG,C18,=B7*LN(1.1),-2.988619,-2.988619,0.0
1,Model 3 LOG-LIN,C18,=EXP(B7*10)-1,-0.010890,-0.010890,0.0
2,Model 3 LOG-LIN,C19,=EXP(B8)-1,-0.080562,-0.080562,0.0
3,Model 3 LOG-LIN,C20,=EXP(B9*10)-1,0.039708,0.039708,0.0
4,Model 3 LOG-LIN,C22,=EXP(B11)-1,0.238444,0.238444,0.0
5,Model 3 LOG-LIN,C23,=EXP(B12)-1,0.044046,0.044046,0.0
6,Model 4 LOG-LOG,C18,=1.1^B7-1,-0.008856,-0.008856,0.0
7,Model 4 LOG-LOG,C19,=EXP(B8)-1,-0.075326,-0.075326,0.0
8,Model 4 LOG-LOG,C20,=1.1^B9-1,0.053473,0.053473,0.0
9,Model 4 LOG-LOG,C22,=EXP(B11)-1,0.242121,0.242121,0.0


**Interpretación.** Excel y Python llegan a los mismos 11 resultados cada quien por su cuenta, así que cada fórmula da el número que dice su oración.

## $\color{#dda}{\text{12. Qué significa esto para el marketplace}}$

La edad es la señal más fuerte, tiene la |t| más grande en los cuatro modelos, y cada año extra va con como 7.5% a 8.1% menos de precio (o 12,940 a 16,979 pesos, según el modelo). Los importados traen un premio de como 24% (unos 42,600 a 44,000 pesos) contra un nacional, y la potencia suma, pero menos que proporcional, con 10% más de hp el precio sube como 5.35%. Los km sí bajan el precio aunque más moderado (10,000 km más son como 1.09% menos, 10% más de km son como 0.89% menos), y ser automático da un premio chiquito de 4.4% a 4.5% (unos 6,400 a 7,400 pesos) contra un manual. El número de dueños anteriores no aparece con evidencia en ningún modelo.

Todo esto es con lo demás igual y habla del precio que piden, no del precio al que el carro se termina vendiendo. Además son asociaciones en datos simulados, así que no prueban causa y efecto, pero para entender qué mueve el precio en este ejercicio por lo menos ya alcanza.